# Sarvam-1 Hindi Continued Pretraining — Colab

Fine-tunes the base **Sarvam-1** model (`sarvamai/sarvam-1`, 2B params, Llama-2 architecture) on Hindi text from **ai4bharat/sangraha** (`verified/hin` config).

Mirrors the repo's `train_finetune_sarvam1.py` script + `run_sarvam1_hindi_baseline_vs_delta.sh` driver, but adapted for **single-GPU Colab** (T4 / A100).

Two modes (toggle in the cell below):
- `baseline` — standard fine-tune, no AttnRes. Apples-to-apples reference.
- `delta_block --null_source` — Delta Block AttnRes with zero-disruption init (recommended, paper default).

Run this notebook **twice** (once per mode) and compare the two `final/` checkpoints.

**Runtime tips**
- Free T4: set `MAX_TOKENS=2_000_000` for a 30–60 min demo run.
- Colab Pro A100 (40/80 GB): bump `MAX_TOKENS=50_000_000` for a meaningful Hindi adaptation.
- Production reference: `run_sarvam1_hindi_baseline_vs_delta.sh` uses 250M tokens over 4 GPUs.

## 1. Mode + hyperparameters

In [ ]:
# ─── Toggle this to switch between baseline and delta_block ───
# Options: "baseline" | "delta_block"
MODE = "delta_block"

USE_NULL_SOURCE = (MODE != "baseline")  # required for zero-disruption init on delta modes
FREEZE_BASE = False                     # set True for LoRA-style: only train AttnRes params

# ─── Hindi continued pretraining target ───
DATASET = "ai4bharat/sangraha"
DATASET_NAME = "verified/hin"
PRETRAINED = "sarvamai/sarvam-1"

# ─── Token budget (cap on total training tokens) ───
# Demo (T4-friendly):       2_000_000  (~30-60 min)
# Meaningful (A100):       50_000_000  (~3-6 hr)
# Production reference:  250_000_000  (the bash script default, multi-GPU)
MAX_TOKENS = 2_000_000

# ─── Training ───
SEQ_LEN = 1024
BATCH_SIZE = 1
GRAD_ACCUM = 16
STEPS = 2000
LR = 3e-4
LR_MIN = 3e-5
LR_ATTNRES = None     # separate LR for AttnRes params (None = same as LR)
WARMUP = 100
MAX_NORM = 1.0
SEED = 42

# ─── Logging / saving ───
USE_WANDB = False     # set True and provide WANDB_API_KEY if you want W&B
WANDB_PROJECT = "residual"
WANDB_ENTITY = "wdlctc_abr"
LOG_EVERY = 10
EVAL_EVERY = 250
EVAL_STEPS = 50
SAVE_EVERY = 1000

## 2. Environment — GPU check + dependency install

Adds the repo's `Attention-Residuals/` directory to `sys.path` so we can import `modeling_sarvam1_attnres`.

In [ ]:
!nvidia-smi | head -20

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"bf16 hardware support: {torch.cuda.is_bf16_supported()}")

In [ ]:
%pip install -q --upgrade transformers datasets accelerate
# versions known to work with the repo (transformers pinned to 5.5.0 in requirements.txt)
%pip install -q "transformers==5.5.0"

In [ ]:
import os, sys

# If running from a fresh clone in Colab, the notebook sits next to Attention-Residuals/
REPO_DIR = "/content/delta-attention-residuals-code" if os.path.exists("/content/delta-attention-residuals-code") else "."
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("sys.path[0:3] =", sys.path[:3])

## 3. Mount Google Drive (for checkpoints)

Skip if you'd rather keep everything in `/content/` (it'll vanish when the runtime disconnects, but downloads are faster).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = f"/content/drive/MyDrive/sarvam1-hin-{MODE}-ft"
print("Checkpoints →", OUT_DIR)

## 4. Imports

In [ ]:
import math, os, time, json
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer, AutoConfig
from transformers.models.llama.modeling_llama import LlamaForCausalLM

from Attention-Residuals.modeling_sarvam1_attnres import (
    Sarvam1AttnResConfig,
    Sarvam1AttnResForCausalLM,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"device={device}  dtype={dtype}")

## 5. W&B (optional)

In [ ]:
use_wandb = False
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = input("Paste your W&B API key: ").strip()
    import wandb
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
               name=f"sarvam1-hin-{MODE}-colab",
               config={"mode": MODE, "max_tokens": MAX_TOKENS, "seq_len": SEQ_LEN,
                       "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
                       "lr": LR, "lr_attnres": LR_ATTNRES, "freeze_base": FREEZE_BASE})
    use_wandb = True

## 6. Build model

- `baseline` → stock `LlamaForCausalLM` from `sarvamai/sarvam-1`.
- `delta_block` (or any AttnRes mode) → `Sarvam1AttnResForCausalLM` with the pretrained weights loaded via `strict=False` (AttnRes-specific params are randomly initialized, then trained).

In [ ]:
def build_model(mode: str, pretrained: str, use_null_source: bool):
    if mode == "baseline":
        model = LlamaForCausalLM.from_pretrained(pretrained, torch_dtype=dtype)
        return model.to(device)

    # 1) Read pretrained config so AttnRes model inherits vocab/hidden/layer dims
    base_cfg = AutoConfig.from_pretrained(pretrained)
    attnres_cfg = Sarvam1AttnResConfig(
        attnres_num_blocks=4,           # matches the repo's default
        attnres_mode=mode,
        attnres_gate_type="bias",
        attnres_use_null_source=use_null_source,
        **{k: v for k, v in base_cfg.to_dict().items()
           if k not in ("model_type", "_name_or_path", "architectures",
                        "auto_map", "transformers_version")},
    )

    # 2) Build AttnRes model with random init for the AttnRes-specific params
    model = Sarvam1AttnResForCausalLM(attnres_cfg)

    # 3) Load pretrained weights (strict=False so AttnRes params stay at their init)
    pretrained_state = LlamaForCausalLM.from_pretrained(
        pretrained, torch_dtype=dtype).state_dict()
    missing, _unexpected = model.load_state_dict(pretrained_state, strict=False)
    attnres_missing = [k for k in missing if "res_" in k or "null_source" in k]
    other_missing = [k for k in missing if k not in attnres_missing]
    if other_missing:
        raise RuntimeError(f"Unexpected missing keys (not AttnRes): {other_missing}")

    return model.to(dtype=dtype, device=device)


print(f"Loading {PRETRAINED} in mode={MODE} …")
model = build_model(MODE, PRETRAINED, USE_NULL_SOURCE)

n_total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total params: {n_total:.1f}M")
if MODE != "baseline":
    n_attnres = sum(p.numel() for n, p in model.named_parameters()
                    if "res_" in n or "null_source" in n)
    print(f"AttnRes params: {n_attnres/1e3:.1f}K ({n_attnres/sum(p.numel() for p in model.parameters())*100:.3f}%)")
    if USE_NULL_SOURCE:
        print("Null source enabled (zero-disruption init)")

## 7. Optional: freeze base, train only AttnRes (LoRA-style)

In [ ]:
if FREEZE_BASE:
    if MODE == "baseline":
        raise ValueError("--freeze_base requires an AttnRes mode, not baseline")
    n_frozen = n_trainable = 0
    for name, p in model.named_parameters():
        if "res_" in name or "null_source" in name:
            p.requires_grad = True
            n_trainable += p.numel()
        else:
            p.requires_grad = False
            n_frozen += p.numel()
    print(f"Freeze base: {n_frozen/1e6:.1f}M frozen, {n_trainable/1e3:.1f}K trainable "
          f"({n_trainable/(n_frozen+n_trainable)*100:.3f}%)")
else:
    for p in model.parameters():
        p.requires_grad = True
    print(f"All params trainable: {n_total:.1f}M")

if dtype == torch.float16:
    # T4 has no native bf16 → use GradScaler for fp16 stability
    scaler = torch.amp.GradScaler("cuda")
    print("fp16 GradScaler enabled (T4 path)")
else:
    scaler = None

## 8. Tokenizer + Hindi data stream

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")

In [ ]:
def hindi_token_stream(dataset_name, config_name, tokenizer, seq_len,
                      max_tokens=None, seed=42):
    """Stream Hindi tokens from Sangraha. Stops after max_tokens (None = run until STEPS)."""
    from datasets import load_dataset
    ds = load_dataset(dataset_name, name=config_name, split="train", streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=10_000)

    buf, produced = [], 0
    for sample in ds:
        if max_tokens is not None and produced >= max_tokens:
            return
        text = sample.get("text") or sample.get("content") or ""
        if not text:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        ids.append(tokenizer.eos_token_id)
        buf.extend(ids)
        while len(buf) >= seq_len + 1:
            if max_tokens is not None and produced >= max_tokens:
                return
            chunk = buf[:seq_len + 1]
            buf = buf[seq_len:]
            produced += seq_len
            yield torch.tensor(chunk, dtype=torch.long)


# Smoke test: pull one chunk
it = hindi_token_stream(DATASET, DATASET_NAME, tokenizer, SEQ_LEN,
                        max_tokens=SEQ_LEN * 2, seed=SEED)
sample_chunk = next(it)
print(f"First chunk: shape={tuple(sample_chunk.shape)}  "
      f"decoded head='{tokenizer.decode(sample_chunk[:30])!r}'")

## 9. Validation — quick WikiText-2 perplexity

In [ ]:
@torch.no_grad()
def eval_wikitext2(model, tokenizer, seq_len, eval_steps):
    from datasets import load_dataset
    model.eval()
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(ds["text"])
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

    nlls, total = [], 0
    for begin in range(0, min(input_ids.size(1), eval_steps * seq_len), seq_len):
        end = min(begin + seq_len, input_ids.size(1))
        chunk = input_ids[:, begin:end]
        with torch.amp.autocast("cuda", dtype=dtype):
            logits = model(input_ids=chunk, use_cache=False).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = chunk[:, 1:].contiguous()
        nll = torch.nn.CrossEntropyLoss(reduction="sum")(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        nlls.append(nll.item())
        total += shift_labels.numel()
    avg_nll = sum(nlls) / total
    model.train()
    return avg_nll, math.exp(avg_nll)

print("Baseline (pre-training) WT2 perplexity:")
init_nll, init_ppl = eval_wikitext2(model, tokenizer, SEQ_LEN, EVAL_STEPS)
print(f"  WT2 loss {init_nll:.4f} | PPL {init_ppl:.2f}")

## 10. Optimizer + scheduler

Cosine with warmup, matching `train_finetune_sarvam1.py`. If `LR_ATTNRES` is set (and mode is AttnRes), uses separate LR groups for base vs. AttnRes params.

In [ ]:
def cosine_with_warmup(step, warmup, total, lr_min_ratio):
    if step < warmup:
        return step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return lr_min_ratio + (1 - lr_min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))


if LR_ATTNRES is not None and MODE != "baseline":
    base_params, attnres_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (attnres_params if ("res_" in name or "null_source" in name) else base_params).append(p)
    optimizer = AdamW(
        [{"params": base_params, "lr": LR},
         {"params": attnres_params, "lr": LR_ATTNRES}],
        betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8)
    scheduler = LambdaLR(optimizer, lr_lambda=[
        lambda s: cosine_with_warmup(s, WARMUP, STEPS, LR_MIN / LR),
        lambda s: cosine_with_warmup(s, WARMUP, STEPS, LR_MIN / LR_ATTNRES),
    ])
    print(f"Optimizer: base LR={LR}, AttnRes LR={LR_ATTNRES}")
else:
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=LR, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: cosine_with_warmup(s, WARMUP, STEPS, LR_MIN / LR))
    print(f"Optimizer: single LR={LR}, {len(trainable)} param tensors")

## 11. Training loop

Mirrors the script's loop: token stream → batch buffer → `grad_accum` micro-batches → step. Logs every `LOG_EVERY`, evaluates on WT2 every `EVAL_EVERY`, saves every `SAVE_EVERY`.

Key Colab adaptations:
- Single GPU, no DDP.
- `torch.amp.autocast` with bf16/fp16.
- `GradScaler` for T4 (fp16).
- Periodic `torch.cuda.empty_cache()` to keep VRAM stable across long runs.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
model.train()
optimizer.zero_grad()

stream = hindi_token_stream(DATASET, DATASET_NAME, tokenizer, SEQ_LEN,
                            max_tokens=MAX_TOKENS, seed=SEED)

global_step = 0
accum_step = 0
accum_loss = 0.0
tokens_seen = 0
t0 = time.time()
batch_buf = []
history = []

print(f"\nTraining {MODE} | max_tokens={MAX_TOKENS:,} | steps={STEPS:,} | "
      f"seq_len={SEQ_LEN} bs={BATCH_SIZE} ga={GRAD_ACCUM}")
print(f"Effective batch tokens/step = {SEQ_LEN * BATCH_SIZE * GRAD_ACCUM:,}")
print("-" * 80)

try:
    for chunk in stream:
        if global_step >= STEPS:
            break

        batch_buf.append(chunk[:-1])
        if len(batch_buf) < BATCH_SIZE:
            continue

        input_ids = torch.stack(batch_buf).to(device)
        labels = input_ids
        batch_buf = []

        with torch.amp.autocast("cuda", dtype=dtype):
            out = model(input_ids=input_ids, labels=labels, use_cache=False)
            loss = out.loss / GRAD_ACCUM

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        accum_loss += loss.item()
        accum_step += 1
        tokens_seen += SEQ_LEN * BATCH_SIZE

        if accum_step < GRAD_ACCUM:
            continue

        if scaler is not None:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
            optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        global_step += 1
        accum_step = 0

        if global_step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            tok_sec = tokens_seen / elapsed
            avg_loss = accum_loss / (LOG_EVERY * GRAD_ACCUM)
            lr_now = scheduler.get_last_lr()[0]
            mem_gb = torch.cuda.max_memory_allocated() / 1e9
            print(f"step {global_step:5d}/{STEPS} | loss {avg_loss:.4f} | "
                  f"lr {lr_now:.2e} | gnorm {grad_norm:.3f} | "
                  f"{tok_sec/1e3:.1f}k tok/s | peak {mem_gb:.1f}GB")
            history.append({"step": global_step, "loss": avg_loss,
                            "lr": lr_now, "tok_per_s": tok_sec})
            if use_wandb:
                wandb.log({"train/loss": avg_loss, "train/lr": lr_now,
                           "train/grad_norm": grad_norm,
                           "train/tok_per_s": tok_sec}, step=global_step)
            accum_loss = 0.0
            tokens_seen = 0
            t0 = time.time()

        if EVAL_EVERY > 0 and global_step % EVAL_EVERY == 0:
            val_loss, val_ppl = eval_wikitext2(model, tokenizer, SEQ_LEN, EVAL_STEPS)
            print(f"  [val] step {global_step} | WT2 loss {val_loss:.4f} | PPL {val_ppl:.2f}")
            if use_wandb:
                wandb.log({"val/wt2_loss": val_loss, "val/wt2_ppl": val_ppl}, step=global_step)
            torch.cuda.empty_cache()

        if global_step % SAVE_EVERY == 0:
            ckpt = os.path.join(OUT_DIR, f"step-{global_step}")
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f"  → saved {ckpt}")
finally:
    # Always save final + history, even if the runtime dies mid-loop
    final_dir = os.path.join(OUT_DIR, "final")
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print(f"\nFinal checkpoint → {final_dir}")
    print(f"History → {os.path.join(OUT_DIR, 'history.json')}")
    if use_wandb:
        wandb.finish()

## 12. Quick generation sanity check

In [ ]:
model.eval()
prompts = [
    "भारत की राजधानी",
    "हिंदी एक",
    "एक समय की बात है,",
]
for prompt in prompts:
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
        out = model.generate(ids, max_new_tokens=40, do_sample=True,
                             temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    print(f">>> {prompt}")
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print()

## 13. What to do with the result

Run this notebook **twice**, toggling `MODE` between `"baseline"` and `"delta_block"` (and changing `OUT_DIR` accordingly). You'll get:

- `OUT_DIR/final/` — loadable with `AutoModelForCausalLM.from_pretrained(...)`.
- `OUT_DIR/history.json` — per-`LOG_EVERY` loss curve. Plot the two runs side-by-side.
- `val/wt2_ppl` (logged every `EVAL_EVERY`) — the standard reference metric.

For a Hindi-specific eval (more meaningful than WT2 for this task), load `OUT_DIR/final` with `lm-eval-harness` using the [`ai2_arc_hindi`](https://huggingface.co/datasets/ai2_arc) and [`indicqa`](https://huggingface.co/datasets/ai4bharat/IndicQA) tasks, or just compare per-step Hindi validation loss on a held-out Sangraha split.

For the **full 250M-token run** with 4 GPUs, use `run_sarvam1_hindi_baseline_vs_delta.sh` from the repo — not this notebook.